<a href="https://colab.research.google.com/github/nihaar06/Bird_species_id/blob/main/notebooks/baseline/Bird_species_id_DL_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

!unzip "/content/drive/MyDrive/logmel_128.zip"

Streaming output truncated to the last 5000 lines.
  inflating: logmel_128/chbchi/XC178009.npy  
  inflating: logmel_128/chbchi/XC193937.npy  
  inflating: logmel_128/chbchi/XC195197.npy  
  inflating: logmel_128/chbchi/XC199587.npy  
  inflating: logmel_128/chbchi/XC201401.npy  
  inflating: logmel_128/chbchi/XC277124.npy  
  inflating: logmel_128/chbchi/XC297464.npy  
  inflating: logmel_128/chbchi/XC325018.npy  
  inflating: logmel_128/chbchi/XC352289.npy  
  inflating: logmel_128/chbchi/XC362937.npy  
  inflating: logmel_128/chbchi/XC362957.npy  
  inflating: logmel_128/chbchi/XC362958.npy  
  inflating: logmel_128/chbchi/XC362959.npy  
  inflating: logmel_128/chbchi/XC363140.npy  
  inflating: logmel_128/chbchi/XC363150.npy  
  inflating: logmel_128/chbchi/XC363200.npy  
  inflating: logmel_128/chbchi/XC363359.npy  
  inflating: logmel_128/chbchi/XC363955.npy  
  inflating: logmel_128/chbchi/XC364337.npy  
  inflating: logmel_128/chbchi/XC364349.npy  
  inflating: logmel_128/chbch

In [ ]:
from tensorflow.keras.layers import MaxPooling2D,Conv2D,Flatten,Dense,Dropout,BatchNormalization,GlobalAveragePooling2D,Activation
from tensorflow.keras.models import Sequential

# def model_cnn(num_classes):
#     cnn=Sequential([
#         Conv2D(16,(3,3),activation='relu',input_shape=(128,313,1)),
#         MaxPooling2D(2,2),
#         Conv2D(32,(3,3),activation='relu'),
#         MaxPooling2D(2,2),
#         Conv2D(64,(3,3),activation='relu'),
#         MaxPooling2D(2,2),
#         Conv2D(128,(3,3),activation='relu'),
#         MaxPooling2D(2,2),
#         Flatten(),
#         Dense(128,activation='relu'),
#         Dropout(0.3),
#         Dense(num_classes,activation='softmax')
#     ])

  # cnn = Sequential([

  #   Conv2D(
  #       16,
  #       (3,3),
  #       padding="same",
  #       input_shape=(128,313,1)
  #   ),
  #   BatchNormalization(),
  #   Activation("relu"),
  #   MaxPooling2D(),

  #   Conv2D(
  #       32,
  #       (3,3),
  #       padding="same"
  #   ),
  #   BatchNormalization(),
  #   Activation("relu"),
  #   MaxPooling2D(),

  #   Conv2D(
  #       64,
  #       (3,3),
  #       padding="same"
  #   ),
  #   BatchNormalization(),
  #   Activation("relu"),
  #   MaxPooling2D(),

  #   Flatten(),

  #   Dense(
  #       128,
  #       activation="relu"
  #   ),

  #   Dropout(0.5),

  #   Dense(
  #       num_classes,
  #       activation="softmax"
  #   )
  #   ])
  #  return cnn
def model_cnn(num_classes):
    cnn = Sequential([
        # Layer 1
        Conv2D(16, (3,3), padding="same", input_shape=(128,313,1)),
        BatchNormalization(),
        Activation("relu"),
        MaxPooling2D((2,2)),

        # Layer 2
        Conv2D(32, (3,3), padding="same"),
        BatchNormalization(),
        Activation("relu"),
        MaxPooling2D((2,2)),

        # Layer 3
        Conv2D(64, (3,3), padding="same"),
        BatchNormalization(),
        Activation("relu"),
        MaxPooling2D((2,2)),

        # Global Pooling replaces Flatten to vastly reduce parameter explosions
        GlobalAveragePooling2D(),

        Dense(128, activation="relu"),
        Dropout(0.4),
        Dense(num_classes, activation="softmax")
    ])
    return cnn

In [ ]:
import tensorflow as tf
from pathlib import Path
import numpy as np

class BirdDataset():
    def __init__(self,root_dir):
        self.root_dir=Path(root_dir)
        self.files=[]
        self.label_map={}
        species_folder=sorted(self.root_dir.iterdir())
        for idx,specie in enumerate(species_folder):
            if not specie.is_dir():
                continue
            specie_name=specie.name
            self.label_map[specie_name]=idx
            for file in specie.glob("*.npy"):
                self.files.append(file)

    def load_sample(self,file_path):
        path=Path(file_path)
        specie_name=path.parent.name
        label=self.label_map[specie_name]
        spec=np.load(path)
        spec=np.expand_dims(spec,axis=-1)
        return spec.astype(np.float32),label

    def get_data(self):
        X=[]
        y=[]
        for file in self.files:
            spec,label=self.load_sample(file)
            X.append(spec)
            y.append(label)
        return np.array(X),np.array(y)


dataset = BirdDataset(
    "logmel_128"
)
X,y=dataset.get_data()
print(X.shape)
print(y.shape)

(8907, 128, 313, 1)
(8907,)


In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

# import os
# print(os.getcwd())
dataset=BirdDataset("logmel_128")
X,y=dataset.get_data()

tf_dataset=tf.data.Dataset.from_tensor_slices(
    (X,y)
)

cnn=model_cnn(num_classes=len(dataset.label_map))
cnn.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 128, 313, 16)   │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 128, 313, 16)   │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 128, 313, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 64, 156, 16)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 64, 156, 32)    │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 64, 156, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (None, 64, 156, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 32, 78, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 32, 78, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 32, 78, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_8 (Activation)       │ (None, 32, 78, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 16, 39, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 52)             │         6,708 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,772 (151.45 KB)

 Trainable params: 38,548 (150.58 KB)

 Non-trainable params: 224 (896.00 B)

In [ ]:
print("Dataset Loaded Successfully!")
print(f"Features shape (X): {X.shape}") # Should show positive values scaled between [0, 1]
print(f"Labels shape (y): {y.shape}")
print(f"Total Unique Classes: {len(dataset.label_map)}")

Dataset Loaded Successfully!
Features shape (X): (8907, 128, 313, 1)
Labels shape (y): (8907,)
Total Unique Classes: 52


In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint
)

callbacks = [

    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),

    ModelCheckpoint(
        "best_bird_model.keras",
        save_best_only=True
    )
]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
import numpy as np

# 1. Stratified Split to ensure validation set has an even representation of rare birds
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Calculate mathematical weights for the imbalanced classes
unique_classes = np.unique(y_train)
utils_weights = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=y_train
)
class_weight_dict = dict(zip(unique_classes, utils_weights))

# 3. Build highly efficient TensorFlow data pipelines
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(len(X_train)).batch(64).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(64).prefetch(tf.data.AUTOTUNE)

cnn.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
history=cnn.fit(train_ds,validation_data=test_ds,epochs=20,callbacks=callbacks)

Epoch 1/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 25s 130ms/step - accuracy: 0.0723 - loss: 3.7814 - val_accuracy: 0.0758 - val_loss: 4.0237
Epoch 2/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - accuracy: 0.1054 - loss: 3.6474 - val_accuracy: 0.1021 - val_loss: 3.6601
Epoch 3/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 6s 54ms/step - accuracy: 0.1217 - loss: 3.5716 - val_accuracy: 0.0303 - val_loss: 4.0873
Epoch 4/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - accuracy: 0.1412 - loss: 3.4660 - val_accuracy: 0.0359 - val_loss: 5.3672
Epoch 5/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - accuracy: 0.1562 - loss: 3.3878 - val_accuracy: 0.0814 - val_loss: 4.1409


In [ ]:
test_loss,test_acc=cnn.evaluate(test_ds)
print("Test Loss:",test_loss)
print("Test Accuracy:",test_acc)

28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2480 - loss: 2.9784
Test Loss: 2.9783968925476074
Test Accuracy: 0.24803590774536133
